# BERTurk Multi-Label Intent Training

This notebook trains `dbmdz/bert-base-turkish-cased` on `deprem-private/intent_test_v13_anonymized`
using the dataset's original multi-label intent annotations.


In [1]:
!pip install -q transformers datasets accelerate scikit-learn pandas numpy


In [2]:
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)


In [3]:
MODEL_NAME = 'dbmdz/bert-base-turkish-cased'
DATASET_NAME = 'deprem-private/intent_test_v13_anonymized'
MAX_LENGTH = 256
TEST_SIZE = 0.15
RANDOM_SEED = 42
THRESHOLD = 0.5


In [4]:
dataset = load_dataset(DATASET_NAME)
dataset


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/539 [00:00<?, ?B/s]

data/train-00000-of-00001-3fe9eb5ff75d67(…):   0%|          | 0.00/314k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2028 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image_url', 'label', 'label_confidence', 'labeler', 'label_creation_time'],
        num_rows: 2028
    })
})

In [5]:
all_labels = set()
for row in dataset['train']:
    for label in row['label']:
        all_labels.add(label)

label_list = sorted(all_labels)
label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for label, idx in label2id.items()}

print('Label sayısı:', len(label_list))
print(label_list)


Label sayısı: 13
['Alakasiz', 'Arama Ekipmani', 'Barınma', 'Cenaze', 'Elektrik Kaynagi', 'Enkaz Kaldirma', 'Giysi', 'Isinma', 'Lojistik', 'Saglik', 'Su', 'Tuvalet', 'Yemek']


In [6]:
def encode_example(example):
    multi_hot = [0.0] * len(label_list)
    for label in example['label']:
        multi_hot[label2id[label]] = 1.0
    example['labels'] = multi_hot
    example['text'] = example['image_url']
    return example

dataset = dataset.map(encode_example)
dataset['train'][0]


Map:   0%|          | 0/2028 [00:00<?, ? examples/s]

{'image_url': '@AlperBicen Kuzenim Ozden  esi [MASK] ve çocukları Odabaşı mah. farklı yaşam rende sitesinde enkaz altindalarOdabaşı  Morgül Sk. No:23 31001 AntakyaLutfen yardim edin bir kaç saat öncesine kadar ses geliyormuş lütfen acill @AFADBaskanlik @ihhafet @Kizilay @BabalaTv @haluklevent ',
 'label': ['Enkaz Kaldirma'],
 'label_confidence': [1.0],
 'labeler': 'system_consensus',
 'label_creation_time': 1676331776163,
 'labels': [0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 'text': '@AlperBicen Kuzenim Ozden  esi [MASK] ve çocukları Odabaşı mah. farklı yaşam rende sitesinde enkaz altindalarOdabaşı  Morgül Sk. No:23 31001 AntakyaLutfen yardim edin bir kaç saat öncesine kadar ses geliyormuş lütfen acill @AFADBaskanlik @ihhafet @Kizilay @BabalaTv @haluklevent '}

In [7]:
split_dataset = dataset['train'].train_test_split(test_size=TEST_SIZE, seed=RANDOM_SEED)
train_ds = split_dataset['train']
val_ds = split_dataset['test']

print(train_ds)
print(val_ds)


Dataset({
    features: ['image_url', 'label', 'label_confidence', 'labeler', 'label_creation_time', 'labels', 'text'],
    num_rows: 1723
})
Dataset({
    features: ['image_url', 'label', 'label_confidence', 'labeler', 'label_creation_time', 'labels', 'text'],
    num_rows: 305
})


In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH,
    )

train_ds = train_ds.map(tokenize_function, batched=True)
val_ds = val_ds.map(tokenize_function, batched=True)

keep_cols = ['input_ids', 'attention_mask', 'labels']
train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep_cols])
val_ds = val_ds.remove_columns([c for c in val_ds.column_names if c not in keep_cols])

train_ds.set_format('torch')
val_ds.set_format('torch')


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1723 [00:00<?, ? examples/s]

Map:   0%|          | 0/305 [00:00<?, ? examples/s]

In [9]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    problem_type='multi_label_classification',
)


model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= THRESHOLD).astype(int)

    return {
        'micro_f1': f1_score(labels, preds, average='micro', zero_division=0),
        'macro_f1': f1_score(labels, preds, average='macro', zero_division=0),
        'samples_f1': f1_score(labels, preds, average='samples', zero_division=0),
        'subset_accuracy': accuracy_score(labels, preds),
    }


In [11]:
training_args = TrainingArguments(
    output_dir='./berturk-deprem-intent',
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=50,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='micro_f1',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to='none',
)


In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)


In [13]:
trainer.train()


Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Samples F1,Subset Accuracy
1,0.197479,0.157315,0.720930,0.305391,0.703065,0.606557
2,0.132294,0.115773,0.819048,0.467211,0.823052,0.708197
3,0.098482,0.104552,0.836707,0.534821,0.840320,0.740984
4,0.085422,0.099026,0.849797,0.561662,0.851746,0.760656


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=864, training_loss=0.1495290816658073, metrics={'train_runtime': 165.0596, 'train_samples_per_second': 41.755, 'train_steps_per_second': 5.234, 'total_flos': 906770244691968.0, 'train_loss': 0.1495290816658073, 'epoch': 4.0})

In [14]:
metrics = trainer.evaluate()
metrics


{'eval_loss': 0.09902583807706833,
 'eval_micro_f1': 0.8497970230040596,
 'eval_macro_f1': 0.5616618760046735,
 'eval_samples_f1': 0.8517460317460317,
 'eval_subset_accuracy': 0.760655737704918,
 'eval_runtime': 1.4099,
 'eval_samples_per_second': 216.324,
 'eval_steps_per_second': 27.661,
 'epoch': 4.0}

In [15]:
trainer.save_model('./berturk-deprem-intent-final')
tokenizer.save_pretrained('./berturk-deprem-intent-final')


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./berturk-deprem-intent-final/tokenizer_config.json',
 './berturk-deprem-intent-final/tokenizer.json')

In [18]:
text = 'Acil çadır ve battaniye ihtiyacı var. Hatay Antakya Odabaşı Mahallesi'
inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=MAX_LENGTH)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

model.eval()
with torch.no_grad():
    logits = model(**inputs).logits
    probs = torch.sigmoid(logits).cpu().numpy()[0]

predicted_labels = [label_list[i] for i, p in enumerate(probs) if p >= THRESHOLD]
print('Tahmin edilen labellar:', predicted_labels)
print('Skorlar:')
for i, p in enumerate(probs):
    if p >= 0.2:
        print(label_list[i], round(float(p), 4))


Tahmin edilen labellar: ['Barınma']
Skorlar:
Barınma 0.9451
Isinma 0.2819
